In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import matplotlib.animation as animation
import matplotlib.colors as colors
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128

from IPython.display import HTML
from astropy.stats import sigma_clip
from tesscube import TESSCube
from astropy.time import Time
from tqdm import tqdm
from tess_ephem import ephem

from tess_asteroids import MovingTPF, __version__
from scipy import stats
from matplotlib import patches
from scipy import signal, ndimage

DATADIR = "/Users/jimartin/Work/TESS/tess-asteroid-ml/data"

In [ ]:
sector = 7
camera = 1
mtpf_size = 21

In [ ]:
def plot_img_aperture(
    img, 
    aperture_mask=None, 
    cbar=True,
    ax=None,
    corner=[0, 0],
    xy=None,
    title=None,
    vmin=None,
    vmax=None,
):

    if ax is None:
        fig, ax = plt.subplots()
        fig.suptitle(f"Asteroid Tracks")

    extent = (
              corner[1] - 0.5,
              corner[1] + img.shape[1] - 0.5,
              corner[0] - 0.5,
              corner[0] + img.shape[0] - 0.5,
    )
    
    im = ax.imshow(
        img, 
        cmap="viridis",
        vmin=vmin, 
        vmax=vmax,
        rasterized=True,
        origin="lower",
        extent=extent,
    )
    if cbar:
        plt.colorbar(im, location="right", shrink=.8, label="Flux [-e/s]")
    if xy is not None:
        dot = ax.scatter(xy[1], xy[0], marker="o", c="red", alpha=1, s=15)
        
    ax.set_aspect('equal', 'box')
    ax.set_title(title)

    if aperture_mask is not None:
        row, col = np.mgrid[corner[0] : corner[0] + img.shape[0], 
                            corner[1] : corner[1] + img.shape[1]]
        for i, pi in enumerate(row[:, 0]):
            for j, pj in enumerate(col[0, :]):
                if aperture_mask[i, j]:
                    # print("here")
                    rect = patches.Rectangle(
                        xy=(pj - 0.5, pi - 0.5),
                        width=1,
                        height=1,
                        color="tab:red",
                        fill=False,
                        # hatch="//",
                        alpha=0.8,
                    )
                    ax.add_patch(rect)

    ax.set_xlabel("Pixel Column")
    ax.set_ylabel("Pixel Row")

    return ax

def animate_image(cube, 
                  aperture_mask=None, 
                  corner=[0, 0], 
                  track=None,
                  cadenceno=None,
                  time=None,
                  interval=200, 
                  repeat_delay=1000
                 ):
    vlo, lo, mid, hi, vhi = np.nanpercentile(cube, [0.2, 1, 50, 95, 99.8])
    
    fig, ax = plt.subplots()
    fig.suptitle(f"Asteroid Tracks in Sector {sector} Camera {camera} CCD {ccd}")

    if aperture_mask is None:
        aperture_mask = np.repeat([None], len(cube), axis=0)
    else:
        if aperture_mask.shape == cube.shape[1:]:
            aperture_mask = np.repeat([aperture_mask], len(cube), axis=0)
            
    if track is None:
        track = np.repeat([None], len(cube), axis=0)
    if cadenceno is None:
        cadenceno = np.repeat([None], len(cube), axis=0)
    if time is None:
        time = np.repeat([None], len(cube), axis=0) 
        
    if len(corner) == 2:
        corner = np.repeat([corner], len(cube), axis=0)

    nt = 0
    ax = plot_img_aperture(cube[nt], 
                           aperture_mask=aperture_mask[nt], 
                           cbar=True,
                           ax=ax,
                           corner=corner[nt],
                           xy=track[nt],
                           title=f"CAD {cadenceno[nt]} | BTJD {time[nt]:.4f}",
                           vmin=0,
                           vmax=10,
                          )

    def animate(nt):
        ax.clear()
        _ = plot_img_aperture(cube[nt], 
                              aperture_mask=aperture_mask[nt], 
                              cbar=False,
                              ax=ax,
                              corner=corner[nt],
                              xy=track[nt],
                              title=f"CAD {cadenceno[nt]} | BTJD {time[nt]:.4f}",
                              vmin=0,
                              vmax=10,
                             )
        
        return ()

    plt.close(ax.figure)

    # Create the animation
    ani = animation.FuncAnimation(
        fig, animate, 
        frames=len(cube), 
        interval=interval, 
        blit=True, 
        repeat_delay=repeat_delay, 
        repeat=True,
    )

    return ani
    

In [ ]:
catalog = pd.read_csv(f"{DATADIR}/jpl/jpl_small_bodies_tess_s{sector:04}-{camera}-0_catalog.csv", index_col=0)
catalog

In [ ]:
plt.hist(catalog.V_mag, bins=100)
plt.show()

In [ ]:
catalog[catalog['RA rate ("/h)'] <= -50].query("V_mag <= 18 and kind == 'a'")

In [ ]:
# sample N bright asteroids
np.random.seed(24)
sample_idx = np.random.choice(catalog.query("V_mag <= 18 and kind == 'a'").index.values, size=20)
sample_idx = np.unique(np.append(sample_idx, catalog[catalog['RA rate ("/h)'] <= -50].query("V_mag <= 18 and kind == 'a'").index.values))
sample_idx

In [ ]:
catalog.columns

In [ ]:
catalog[['V_mag', 'Inclination (deg)', 'RA rate ("/h)', 'Dec rate ("/h)']].iloc[sample_idx]

In [ ]:
# get TPFs with tess-asteroids
mtpf_dir = f"{DATADIR}/moving_TPFs/sector{sector:04}"

for idx in tqdm(sample_idx[15:]):
    try:
        target, df_ephem = MovingTPF.from_name(catalog.iloc[idx].id, sector=sector)
    except NotImplementedError:
        print("Multiple CCDs")
        df_ephem = ephem(catalog.iloc[idx].id, sector=sector, time_step=1.0)
        print(np.unique(df_ephem.camera.values))
        print(np.unique(df_ephem.ccd.values))
        # break
        target, df_ephem = MovingTPF.from_name(catalog.iloc[idx].id, 
                                            sector=sector, 
                                            camera=np.unique(df_ephem.camera.values)[1],
                                            ccd=np.unique(df_ephem.ccd.values)[1]
                                           )
    except ValueError:
        print("Not Observed!!!!", catalog['RA rate ("/h)'].iloc[idx])
        continue
    try:
        target.get_data(shape=(mtpf_size, mtpf_size))
    except RuntimeError:
        print("Pixels out of bound")
        continue
    # continue
    target.reshape_data()
    target.background_correction(method="rolling", nframes=31)
    target.create_pixel_quality()
    target.create_aperture(method="prf")
    hdul, fname = target._save_tpf(file_name=None, save_loc=mtpf_dir, return_hdu=True)
    hdul[0].header.set("DATE", datetime.now().strftime("%Y-%m-%d"), comment="file creation date")
    hdul[0].header.set("CREATOR", "tess-asteroids", comment="software used to produce this file")
    hdul[0].header.set("CREATOR", __version__, comment="software version")
    
    hdul[0].header.set("TSTART", target.time[0], comment="observation start time in TJD of first frame")
    hdul[0].header.set("TSTOP", target.time[-1], comment="observation start time in TJD of last frame")
    hdul[0].header.set("DATE-OBS", Time(target.time[0], format="btjd", scale="tdb").utc.isot, comment="TSTART as UTC calendar date")
    hdul[0].header.set("DATE-END", Time(target.time[-1], format="btjd", scale="tdb").utc.isot, comment="TSTOP as UTC calendar date")
    
    hdul[0].header.set("OBJECT", catalog.iloc[idx]["Object name"], comment="Asteroid name")
    hdul[0].header.set("VMAG", catalog.iloc[idx]["V_mag"], comment="V magnitude")
    hdul[0].header.set("HMAG", catalog.iloc[idx]["H_mag"], comment="H absolute magnitude")
    hdul[0].header.set("PERIHEL", catalog.iloc[idx]["Perihelion (au)"], comment="Perihelion (au)")
    hdul[0].header.set("ORBECC", catalog.iloc[idx]["Eccentricity"], comment="Orbit eccentricity")
    hdul[0].header.set("ORBINC", catalog.iloc[idx]["Inclination (deg)"], comment="Orbit inclination [deg]")
    hdul[0].header.set("RARATE", catalog.iloc[idx]['RA rate ("/h)'], comment='Right ascension rate ["/h]')
    hdul[0].header.set("DECRATE", catalog.iloc[idx]['Dec rate ("/h)'], comment='Declination rate ["/h]')

    hdul.writeto(fname, overwrite=True)
    
    # break

In [ ]:
ephem

In [ ]:
catalog['RA rate ("/h)'].iloc[idx]

In [ ]:
nt = 200

plot_img_aperture(
    target.corr_flux[nt], 
    aperture_mask=target.aperture_mask[nt], 
    # ax=ax,
    cbar=True,
    corner=target.corner[nt],
    xy=target.ephemeris[nt],
    title=f"CAD {target.cadence_number[nt]} | BTJD {target.time[nt]:.4f}",
    vmin=0,
    vmax=10,
    )

In [ ]:
ccd = target.ccd
HTML(
    animate_image(target.corr_flux, 
                  aperture_mask=target.aperture_mask, 
                  corner=target.corner, 
                  track=target.ephemeris,
                  cadenceno=target.cadence_number,
                  time=target.time,
                  interval=200, 
                  repeat_delay=1000
                 ).to_jshtml()
)